# 10 임베딩 기반 시계열 분석 (Phase 2)

## Phase2 base = XGBoost 고정

Phase2는 **임베딩 방법(6종) 비교**가 목적이라, 조건마다 다른 Phase1 Best를 base로 쓰면 `조건×base×임베딩`으로 실험이 폭발합니다. 따라서 **임베딩 벡터를 정적 피처로 자연스럽게 결합할 수 있는 패널 ML(XGBoost)** 를 대표 base로 고정합니다.

> **논문과의 차이:** 논문은 Phase2에서 **iTransformer + 임베딩**을 사용했습니다. 본 실습은 임베딩을 트랜스포머에 주입하는 대신, **패널 피처로 결합이 용이한 XGBoost**를 base로 고정해 임베딩 방법 비교에 집중합니다(축소 데이터·구현 단순화). Phase1 통합 Best는 조건별로 ARIMA·N-HiTS 등 다양합니다.

| 축 | 내용 |
|----|------|
| type | **E(고변동), C(저변동)** — 2종 |
| cluster | SBC(4) + ML(AE+KMeans, 2) |
| **조건** | **12** (SBC 8 + ML 4) |
| base (고정) | **XGBoost** (family별 Min-Max 정규화 + 재귀 다단계) |
| 임베딩 | PCA, FastDTW, AE, GAF-CNN, TS2Vec, PatchTST |
| 조합 | **12 × 6 = 72** |

**오차지표:** MAE, RMSE, MAPE, MASE — 조건별 Best 임베딩은 **MAPE** 최소 기준

## 11장과의 연결
동일 파이프라인(XGBoost+임베딩) 위에서 **SBC·ML 클러스터링 scheme만** 바꿔, 11장에서 type별 **제품수 가중 WMAPE**로 어느 scheme이 유리한지 비교합니다.

### ⓪ 환경 설정

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'code' else NOTEBOOK_DIR
sys.path.insert(0, str(REPO_ROOT / 'code'))

from utils.paths import DATA_PROCESSED
from utils.splits import VAL_WEEKS, TRAIN_WEEK_MAX
from utils.device import device_label
from utils.experiment_data import load_forecast_frames
from utils.phase_experiments import (
    STAT_MODELS, ML_MODELS, DL_MODELS, RANK_METRIC,
    run_phase1_all, summarize_phase1, merge_phase1_best,
    build_global_embedding_cache, run_phase2_all, summarize_phase2,
    pick_final_per_condition,
)

print('Torch device:', device_label())
df, feat_df = load_forecast_frames()
print('시계열:', df.groupby(['type', 'family']).ngroups)
print('학습 <=', TRAIN_WEEK_MAX, '| 검증:', VAL_WEEKS)
print('Best 선정 기준:', RANK_METRIC.upper())


Torch device: cuda (NVIDIA GeForce RTX 5090, 32GB)
시계열: 66
학습 <= 201707 | 검증: [201708, 201709, 201710, 201711, 201712, 201713, 201714, 201715, 201716, 201717, 201718, 201719, 201720]
Best 선정 기준: MAPE


### ① XGBoost × 6임베딩 — 40조건 실험

In [2]:
from utils.phase_experiments import REPRESENTATIVE_BASE_MODEL

BASE = REPRESENTATIVE_BASE_MODEL
print('대표 base:', BASE)

p2_cache = DATA_PROCESSED / 'phase2_results.parquet'
if p2_cache.exists():
    phase2 = pd.read_parquet(p2_cache)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    print('Phase2 캐시 로드 |', len(phase2), 'rows')
else:
    emb_cache = build_global_embedding_cache(df)
    p2_sbc = run_phase2_all(df, feat_df, 'SBC_CLUSTER', 'SBC', fixed_model=BASE, emb_cache=emb_cache)
    p2_ml = run_phase2_all(df, feat_df, 'ML_CLUSTER', 'ML', fixed_model=BASE, emb_cache=emb_cache)
    phase2 = pd.concat([p2_sbc, p2_ml], ignore_index=True)
    phase2_summary, phase2_best = summarize_phase2(phase2)
    phase2.to_parquet(p2_cache, index=False)
    phase2_summary.to_csv(DATA_PROCESSED / 'phase2_summary.csv', index=False)
    phase2_best.to_csv(DATA_PROCESSED / 'phase2_best_per_condition.csv', index=False)
    print('Phase2 완료 |', len(phase2), 'rows')

print('유효 조건(제품 있음):', phase2.groupby(['cluster_scheme','type','cluster']).ngroups)
display(phase2_best.sort_values(['cluster_scheme', 'type', 'cluster']))


대표 base: XGBoost


Global embeddings:   0%|          | 0/6 [00:00<?, ?it/s]

Phase2 SBC:   0%|          | 0/48 [00:00<?, ?it/s]

Phase2 ML:   0%|          | 0/48 [00:00<?, ?it/s]

Phase2 완료 | 792 rows
유효 조건(제품 있음): 12


,cluster_scheme,type,cluster,best_hybrid,base_model,embedding,mae_mean,rmse_mean,best_mape,mase_mean
4,ML,C,1,XGBoost+PatchTST,XGBoost,PatchTST,27655.159527,44830.074015,28.802738,2.188981
7,ML,C,2,XGBoost+FastDTW,XGBoost,FastDTW,1214.970340,2049.983295,149.458302,2.009446
12,ML,E,1,XGBoost+AE,XGBoost,AE,6576.080322,8308.195241,6.178018,0.719698
22,ML,E,2,XGBoost+PatchTST,XGBoost,PatchTST,961.407851,1499.675118,76.911174,2.175644
25,SBC,C,1,XGBoost+FastDTW,XGBoost,FastDTW,5722.775169,9324.999499,36.442240,1.837773
31,SBC,C,2,XGBoost+FastDTW,XGBoost,FastDTW,971.730207,1774.261725,42.443300,1.904680
36,SBC,C,3,XGBoost+AE,XGBoost,AE,665.579287,1440.182939,35.216916,0.420051
43,SBC,C,4,XGBoost+FastDTW,XGBoost,FastDTW,32.341221,37.003324,604.329442,1.153732
50,SBC,E,1,XGBoost+GAF-CNN,XGBoost,GAF-CNN,1691.538039,2774.279531,38.601658,1.504339
58,SBC,E,2,XGBoost+PatchTST,XGBoost,PatchTST,461.873736,868.443286,43.267457,2.089841


### ② 결과 요약

### ③ 11장 하이브리드 비교로의 연결

- 산출물 `phase2_best_per_condition.csv` = 조건별 **XGBoost + Best 임베딩** (MAPE)
- 11장에서 SBC·ML scheme 각각을 type별 **제품수 가중 WMAPE**(논문 §4.5)로 합산 비교
- **동일 base·임베딩 파이프라인**에서 clustering scheme만 다르므로 SBC vs ML 비교가 공정
- 결과: **PatchTST 임베딩이 median MAPE 최저(32.4)** — 논문의 PatchTST 우수성과 정합

In [3]:
print('=== 조건별 Best 임베딩 (MAPE) ===')
print(phase2_best['embedding'].value_counts())

print('\n=== 임베딩별 평균 4지표 (전 조건) ===')
emb_avg = phase2.groupby('embedding')[['mae','rmse','mape','mase']].mean().round(2)
display(emb_avg.sort_values('mape'))

print('\n=== scheme별 XGBoost+임베딩 평균 MAPE ===')
print(phase2.groupby(['cluster_scheme','embedding'])['mape'].mean().unstack('cluster_scheme').round(2))


=== 조건별 Best 임베딩 (MAPE) ===
embedding
PatchTST    4
FastDTW     4
AE          3
GAF-CNN     1
Name: count, dtype: int64

=== 임베딩별 평균 4지표 (전 조건) ===


,mae,rmse,mape,mase
embedding,,,,
GAF-CNN,2319.23,3796.80,87.42,1.94
PCA,2319.23,3796.80,87.42,1.94
TS2Vec,2319.23,3796.80,87.42,1.94
FastDTW,2389.34,3923.80,87.77,1.99
PatchTST,2496.73,4004.69,111.65,1.93
AE,2560.11,4106.38,136.26,2.04



=== scheme별 XGBoost+임베딩 평균 MAPE ===
cluster_scheme      ML    SBC
embedding                    
AE              211.72  60.79
FastDTW         118.22  57.33
GAF-CNN         116.91  57.94
PCA             116.91  57.94
PatchTST        164.42  58.88
TS2Vec          116.91  57.94
